## Phase 1: Predicting Drug resistance (y = Drugs, X = Mutations)

In [58]:
#Import packages
import pandas as pd

#Read in data
df = pd.read_csv("geno-pheno.dataset.tsv", sep = "\t")

df.head()

/tmp/ipykernel_8898/2676976629.py:5: DtypeWarning: Columns (0: AZT, 1: DDCFoldMatch, 2: TAFFoldMatch) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("geno-pheno.dataset.tsv", sep = "\t")


,RefID,IsolateID,IsolateName,Species,Type,PtID,Method,3TC,3TCFoldMatch,ABC,...,P291,P292,P293,P294,P295,P296,P297,P298,P299,P300
0,756,9918,CA9918,HIV1,Clinical,1391.0,PhenoSense,200.0,>,4.6,...,-,-,-,-,-,-,-,-,-,-
1,756,3832,CA3832,HIV1,Clinical,1433.0,PhenoSense,200.0,>,8.8,...,-,-,.,.,.,.,.,.,.,.
2,756,10464,CA10464,HIV1,Clinical,634.0,PhenoSense,200.0,>,14.0,...,.,.,.,.,.,.,.,.,.,.
3,756,9928,CA9928,HIV1,Clinical,637.0,PhenoSense,200.0,>,6.7,...,-,-,V,-,-,-,K,-,-,-
4,756,4372,CA4372,HIV1,Clinical,1274.0,PhenoSense,200.0,>,7.1,...,-,-,V,-,-,-,-,-,-,-


In [64]:
##Cleaning the data set
df2 = df.copy()
df2.head()

,RefID,IsolateID,IsolateName,Species,Type,PtID,Method,3TC,3TCFoldMatch,ABC,...,P291,P292,P293,P294,P295,P296,P297,P298,P299,P300
0,756,9918,CA9918,HIV1,Clinical,1391.0,PhenoSense,200.0,>,4.6,...,-,-,-,-,-,-,-,-,-,-
1,756,3832,CA3832,HIV1,Clinical,1433.0,PhenoSense,200.0,>,8.8,...,-,-,.,.,.,.,.,.,.,.
2,756,10464,CA10464,HIV1,Clinical,634.0,PhenoSense,200.0,>,14.0,...,.,.,.,.,.,.,.,.,.,.
3,756,9928,CA9928,HIV1,Clinical,637.0,PhenoSense,200.0,>,6.7,...,-,-,V,-,-,-,K,-,-,-
4,756,4372,CA4372,HIV1,Clinical,1274.0,PhenoSense,200.0,>,7.1,...,-,-,V,-,-,-,-,-,-,-


In [65]:
# Dimensionality of data
df2.shape

(2505, 335)

### Exploratory data analysis

In [66]:
#DRop columns we don't need in the training
df2 = df.drop(columns = ["RefID","Species","Type", "Method", "NNRTIDRMs", 'CompleteMutationListAvailable', 'Author','NonDRMs', 'Author','RefYear', 'MedlineID', 'Title',  'PtID'])
df2.head(n = 300)

#Remove Protease positions

df2 = df2.loc[:, ~df2.columns.str.match(r"^P\d+$")]
df2.head()
df2.shape

(2505, 23)

In [67]:
# Explore the data
#df2.describe()

#df2.info()

df2.columns

df2.shape

(2505, 23)

In [68]:
# Find reverse transcriptase columns
[col for col in df2.columns if "RT" in col.upper()]

['NRTIDRMs']

In [69]:
#Check which columns retained
df2.columns

Index(['IsolateID', 'IsolateName', '3TC', '3TCFoldMatch', 'ABC',
       'ABCFoldMatch', 'AZT', 'AZTFoldMatch', 'D4T', 'D4TFoldMatch', 'DDC',
       'DDCFoldMatch', 'DDI', 'DDIFoldMatch', 'FTC', 'FTCFoldMatch', 'ISL',
       'ISLFoldMatch', 'TAF', 'TAFFoldMatch', 'TDF', 'TDFFoldMatch',
       'NRTIDRMs'],
      dtype='str')

In [70]:
#View data 
df2.head()

,IsolateID,IsolateName,3TC,3TCFoldMatch,ABC,ABCFoldMatch,AZT,AZTFoldMatch,D4T,D4TFoldMatch,...,DDIFoldMatch,FTC,FTCFoldMatch,ISL,ISLFoldMatch,TAF,TAFFoldMatch,TDF,TDFFoldMatch,NRTIDRMs
0,9918,CA9918,200.0,>,4.6,=,0.6,=,1.0,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M184V
1,3832,CA3832,200.0,>,8.8,=,3.2,=,1.9,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,>,14.0,=,307,=,6.4,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,>,6.7,=,6.5,=,1.6,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,>,7.1,=,0.8,=,1.3,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"D67N, K70G, M184V, T215F, K219Q"


In [71]:
#Identifying unique mutations in the dataset

df2.dtypes

IsolateID         int64
IsolateName         str
3TC             float64
3TCFoldMatch        str
ABC             float64
ABCFoldMatch        str
AZT              object
AZTFoldMatch        str
D4T             float64
D4TFoldMatch        str
DDC             float64
DDCFoldMatch        str
DDI             float64
DDIFoldMatch        str
FTC             float64
FTCFoldMatch        str
ISL             float64
ISLFoldMatch        str
TAF             float64
TAFFoldMatch        str
TDF             float64
TDFFoldMatch        str
NRTIDRMs            str
dtype: object

In [72]:
# Number of duplicate IsolateIDs
print("Duplicate IsolateIDs:", df2["IsolateID"].duplicated().sum())


Duplicate IsolateIDs: 0


In [73]:
# Abstract drugs
missing = (df2.isnull().sum().to_frame(name="Missing"))

missing["Percent"] = round(missing["Missing"] / len(df_clean) * 100, 2)

missing.sort_values("Percent", ascending=False)

,Missing,Percent
ISLFoldMatch,2473,98.72
ISL,2473,98.72
TAFFoldMatch,2409,96.17
TAF,2409,96.17
DDCFoldMatch,2036,81.28
DDC,2003,79.96
FTCFoldMatch,1948,77.76
FTC,1948,77.76
NRTIDRMs,512,20.44
TDF,493,19.68


In [74]:
#Measure completeness of Drug data
drug_cols = ["3TC", "ABC", "AZT", "D4T", "DDC", "DDI", "FTC", "ISL", "TAF", "TDF"]

available = df2[drug_cols].notna().sum().sort_values(ascending=False)

print(available)

AZT    2381
D4T    2377
DDI    2377
3TC    2359
ABC    2231
TDF    2012
FTC     557
DDC     502
TAF      96
ISL      32
dtype: int64


In [76]:
#DRop columns we don't need in the training
df_clean = df2.drop(columns = ["3TCFoldMatch", "ABCFoldMatch", "AZTFoldMatch", "D4TFoldMatch", "DDCFoldMatch", "DDIFoldMatch", "FTCFoldMatch", "ISLFoldMatch", "TAFFoldMatch", "TDFFoldMatch"])
df_clean.head(n = 300)

#Drop DRugs with < 1000 complete cells
df_clean = df_clean.drop(columns = ["DDC", "FTC", "ISL", "TAF"])
df_clean.head(n=30)



,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W"
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y"


In [77]:
#Data types for df_clean
print(df_clean.dtypes)

IsolateID        int64
IsolateName        str
3TC            float64
ABC            float64
AZT             object
D4T            float64
DDI            float64
TDF            float64
NRTIDRMs           str
dtype: object


In [79]:
#Save cleaned data 
df_clean.to_csv("geno-pheno_clean.tsv", sep="\t", index=False)
df_clean.head(n=30)

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W"
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y"


In [80]:
# Remove Isolates that lack NRTIDRMs
#df_clean.dropna(subset = ["NRTIDRMs"], inplace= True)
#df_clean

#Replacing isolates that lack NRTIDRMs with empty strings
df_clean["NRTIDRMs"] = df_clean["NRTIDRMs"].fillna("")
df_clean.tail(n = 30)

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
2475,743974,DOR-clinical-resistance-7,100.0,2.9,0.5,0.8,1.6,0.8,M184V
2476,692618,ISO3,1.2,0.8,1.7,0.9,0.9,1.2,
2477,692622,ISO7,0.9,1.5,1.5,1.4,1.6,1.8,
2478,692619,ISO4,0.4,0.7,0.3,0.6,0.7,0.3,
2479,743968,DOR-clinical-resistance-1,100.0,2.7,0.1,0.7,1.2,0.5,M184V
2480,743969,DOR-clinical-resistance-2,100.0,3.2,0.1,0.6,1.2,0.6,M184V
2481,692621,ISO6,0.4,0.7,0.8,1.0,0.1,0.8,
2482,743971,DOR-clinical-resistance-4,3.1,0.7,0.2,0.7,1.0,0.3,
2483,692616,ISO1,0.4,1.7,0.3,1.3,1.1,1.5,
2484,743973,DOR-clinical-resistance-6,100.0,2.8,0.1,0.5,1.5,0.4,"K65R, M184V"


In [81]:
#Count Number of Isolates - Number reduced after pruning isolates lacking NRTI-DRMs
df_clean["IsolateID"].nunique()

2505

In [83]:
## Mutation List Generation
df_clean['Mutation_List'] = df_clean['NRTIDRMs'].apply(lambda s: [m.strip() for m in s.split(',') if m.strip()])
df_clean["Mutation_List"].head(n=30)
df_clean["Mutation_List"].tail(n = 30)

2475                                              [M184V]
2476                                                   []
2477                                                   []
2478                                                   []
2479                                              [M184V]
2480                                              [M184V]
2481                                                   []
2482                                                   []
2483                                                   []
2484                                        [K65R, M184V]
2485                                              [M184V]
2486                                               [K65R]
2487                                                   []
2488                                                   []
2489    [M41L, E44D, D67N, T69D, L74V, M184V, L210W, T...
2490                                                   []
2491                           [M41L, L74V, L210W, T215Y]
2492    [M41ML

### Calculating Mutation Frequencies

In [84]:
# Preparing for features (Mutations) Matrix generation 
df_clean[["IsolateName","NRTIDRMs", "Mutation_List"]].tail(30)


,IsolateName,NRTIDRMs,Mutation_List
2475,DOR-clinical-resistance-7,M184V,[M184V]
2476,ISO3,,[]
2477,ISO7,,[]
2478,ISO4,,[]
2479,DOR-clinical-resistance-1,M184V,[M184V]
2480,DOR-clinical-resistance-2,M184V,[M184V]
2481,ISO6,,[]
2482,DOR-clinical-resistance-4,,[]
2483,ISO1,,[]
2484,DOR-clinical-resistance-6,"K65R, M184V","[K65R, M184V]"


### Feature Matrix Generation

In [93]:
# Mutation frequency calculation
#Expanding comma-separated mutations into separate rows (Long Format)
df_mut_exploded = df_clean.explode('Mutation_List').dropna(subset=['Mutation_List'])
df_mut_exploded.head()

#Mutation Frequency Analysis
total_isolates = df_clean['IsolateID'].nunique()
global_frequencies = df_mut_exploded['Mutation_List'].value_counts().reset_index()
global_frequencies = global_frequencies[
    (global_frequencies['Mutation_List'] != "") 
]
global_frequencies.columns = ['Mutation_List', 'Absolute_Count']

global_frequencies. head(n = 30)
#print(total_isolates)


,Mutation_List,Absolute_Count
0,M184V,1153
1,M41L,999
2,T215Y,887
3,D67N,811
4,L210W,697
5,K70R,474
6,K219Q,364
7,T215F,259
8,T69D,241
9,E44D,211


In [94]:
#Check dimensionality
df_clean.columns

#Check empty
mut_empty = (df_clean["Mutation_List"].str.len() == 0).sum()
mut_empty


512

total_isolate

In [95]:
#Calculate proportionality of mutations
global_frequencies['Global_Frequency_%'] = (global_frequencies['Absolute_Count'] / (total_isolates - mut_empty))* 100
global_frequencies.head(30)


,Mutation_List,Absolute_Count,Global_Frequency_%
0,M184V,1153,57.852484
1,M41L,999,50.125439
2,T215Y,887,44.505770
3,D67N,811,40.692423
4,L210W,697,34.972403
5,K70R,474,23.783241
6,K219Q,364,18.263924
7,T215F,259,12.995484
8,T69D,241,12.092323
9,E44D,211,10.587055


In [ ]:
# Filter based on mutation frequency (< 0.4)
retained_mutations = global_frequencies[global_frequencies["Global_Frequency_%"] >= 1.0]
retained_mutations.head()
retained_mutations.tail()

,Mutation_List,Absolute_Count,Global_Frequency_%
37,A62AV,27,1.354742
38,K219KR,27,1.354742
39,K219KN,26,1.304566
40,T215D,23,1.154039
41,K70G,22,1.103864


In [97]:
retained_mutations.shape

(42, 3)

## Subsetting data 

In [98]:
#Data - df_clean
df_clean.head(n=30)

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs,Mutation_List
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V,[M184V]
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y","[M41L, E44A, D67N, T69D, M184V, L210W, T215Y]"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y","[M41L, M184V, T215Y]"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q","[D67N, K70G, M184V, T215F, K219Q]"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q","[D67N, K70R, M184V, K219Q]"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...","[E40F, M41L, D67N, V75M, M184V, L210W, T215Y, ..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W","[M41L, D67N, T69D, K70R, V75M, M184V, T215F, K..."
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q","[D67N, K70R, M184V, K219Q]"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y","[M41L, M184V, L210W, T215Y]"


In [101]:
# Fix AZT fold change values, currently object (str) ---> float
df_clean["AZT"] = (df_clean["AZT"].astype(str).str.replace(",","", regex = False).replace ('nan', pd.NA))
df_clean["AZT"] = pd.to_numeric(df_clean["AZT"], errors = "coerce")

df_clean["AZT"].dtypes

dtype('float64')

In [102]:
# Drug Thresholds 
DRUGS = {
    '3TC' : 3.5,
    'ABC' : 4.5,
    'AZT' : 1.9,
    'D4T' : 1.7,
    'DDI' : 1.3,
    'TDF' : 1.4
}

In [103]:
df_clean.columns

Index(['IsolateID', 'IsolateName', '3TC', 'ABC', 'AZT', 'D4T', 'DDI', 'TDF',
       'NRTIDRMs', 'Mutation_List'],
      dtype='str')

In [106]:
#### 3TC subsetting
print("Lamivudine (3TC) subsetting .....")

df_3TC = (df_clean[['IsolateID','NRTIDRMs', '3TC']].rename(columns = {'3TC' : "FoldChange"}).dropna(subset = ["FoldChange"]).reset_index(drop = True))
df_3TC["resistance"] = (df_3TC["FoldChange"]>= 3.5).astype(int)
df_3TC.head()

Lamivudine (3TC) subsetting .....


,IsolateID,NRTIDRMs,FoldChange,resistance
0,9918,M184V,200.0,1
1,3832,"M41L, L74LV, M184V, L210W, T215Y",200.0,1
2,10464,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y",200.0,1
3,9928,"M41L, M184V, T215Y",200.0,1
4,4372,"D67N, K70G, M184V, T215F, K219Q",200.0,1


In [ ]:
# ## Create the feature matrix (X) - One Hot encoding of the 'Mutation_List

# from sklearn.preprocessing import MultiLabelBinarizer

# #Create multilabelBinarizer object
# mlb = MultiLabelBinarizer()

# # One-Hot encode ["Mutation_List"]
# X = mlb.transform(df_clean["Mutation_List"])

# X = pd.DataFrame(X, columns= mlb.classes_, index=df_clean["IsolateID"])

# X.head(n=30)

,,A62AV,A62V,D67DEG,D67DG,D67DH,D67DN,D67E,D67EK,D67G,...,V75M,V75MT,V75S,V75T,V75VA,V75VI,V75VIM,V75VM,Y115F,Y115YF
IsolateID,,,,,,,,,,,,,,,,,,,,,
9918,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3832,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10464,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9928,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4372,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4391,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9912,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
9945,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
9916,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# # Explore Feature Matrix (X)
# print(X.shape)

(2505, 166)


In [ ]:
# #Inspect feature name (Make sure they are Mutations)
# X.columns.tolist()[:20]

['',
 'A62AV',
 'A62V',
 'D67DEG',
 'D67DG',
 'D67DH',
 'D67DN',
 'D67E',
 'D67EK',
 'D67G',
 'D67GS',
 'D67GV',
 'D67H',
 'D67HN',
 'D67N',
 'D67NH',
 'D67NS',
 'D67NT',
 'D67S',
 'D67~']

### Target Matrix Generation

In [ ]:
# df_target = pd.read_csv("geno-pheno_clean.tsv",sep = "\t")
# df_target.head(n = 30)
# df_target['Drug_List'] = df_target['Drugs'].apply(lambda s: [d.strip() for d in s.split(',') if d.strip()])

KeyError: 'Drugs'

In [ ]:
# Create y - dataframe
# df_target = df_clean[["IsolateID","3TC","ABC","AZT","D4T","DDI","TDF"]]
# y.head(n = 30)

KeyError: 'drugs'

In [ ]:
# Countercheck Isolate_Name compatibility between X and y

# X = X.loc[y.index]

NameError: name 'y' is not defined